In [ ]:
from platform import python_version
print(python_version())

### Cluster with Tahoe

### Huggingface: tahoebio/Tahoe-x1-embeddings

https://github.com/tahoebio/tahoe-x1

Tahoe-x1: Scaling Perturbation-Trained Single-Cell Foundation Models to 3 Billion Parameters


#### Memory

That's not a general "64 GB isn't enough" — swap is fully exhausted at 2.0G, which means something asked for tens of GB in one allocation. 

Given where you are in the pipeline, the culprit is almost certainly load_tahoe_de, and the arithmetic says so:

The DE table is ~4.09e9 rows over ~75k conditions × ~54k genes. 

Filtering to pancreas doesn't help much — roughly 
- 6 lines × 379 drugs × ~4 doses × 54k genes ≈ 5e8 rows, 
- materialised in pandas with gene/drug/cell_line_id as object-dtype strings (~200 B/row) before pivot_table ever runs. 
- That's >100 GB. full_Z and consensus_cluster are megabytes by comparison.



In [ ]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, ROOT_SRC)


if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import create_dir
from libs.MTD_lib import MTD
from libs.cBioPortal_lib import cBioPortal
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config
from libs.prism_lib import PRISM
from libs.prism_program_lib import *


from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

In [ ]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'BRCA'
PSI_ID = 'ACC'
PSI_ID = 'CESC'
PSI_ID = 'PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

In [ ]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", mtd.disease, case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

In [ ]:
cbio = cBioPortal(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [ ]:
verbose = False

df_psi = cbio.open_primary_site(verbose=verbose)
df_psi

### Open primary cites from cbio

In [ ]:
PROG_ID = 'TCGA'
psi_id = 'PAAD'
psi_id = 'SKCM'
psi_id = 'BRCA'

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'

### Prism - development

In [ ]:
import anndata as ad

prism = PRISM(root0=ROOT0, root0_data=ROOT0_DATA)

verbose=True

prism.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)

prism.root_singc, prism.root_singc.exists()

### Running prism

In [ ]:
verbose=True

res = prism.open_bayesprism(verbose=verbose)
print(len(res.genes))

In [ ]:
res.states

In [ ]:
res.theta_type.columns

In [ ]:
print(res.theta.shape)
res.theta.head(5)

In [ ]:
res.theta.tail(5)

### Ductal cell type 1

"Ductal cell type 1" is the normal-like ductal population and stays in the environment compartment — which is what you want. If both had been mapped to malignant, purity would inflate. Verify with res.tumor_purity.groupby(meta["condition"]).describe(): normals near zero, tumors somewhere in 0.2–0.6.


### Ductal cell type 2

One malignant state means no Ductal cell type 2 subdivision, so subtype_malignant scores Moffitt signatures on a single pooled malignant profile. That still works — it's per-sample expression, so samples can differ — but it won't give you distinct malignant states in θ. For that you'd subcluster Ductal cell type 2 in the AnnData and write finer cell_state labels before calling pseudobulk_reference.

In [ ]:
res.cell_type_expression("Ductal cell type 1").shape

In [ ]:
res.cell_type_expression("Ductal cell type 1").head(3)

In [ ]:
res.cell_type_expression("Ductal cell type 2").shape

In [ ]:
res.cell_type_expression("Ductal cell type 2").head(3)

### 2. theta is now fixed -> expand Z to every gene

In [ ]:
verbose=False
force=False

imax_tumor=250
imax_normal=50

exclude_prog_list=['CCLE']
disease_cd = 'PAAD'

dfn_tumor, dfn_normal, df_gtex, df_summ = cbio.get_all_data_from_disease(disease_cd=disease_cd, 
                                                           imax_tumor=imax_tumor, imax_normal=imax_normal,
                                                           exclude_prog_list=exclude_prog_list,
                                                           force=force, verbose=verbose)

df_bulk, df_meta = prism.build_bulk_matrix(dfn_tumor, dfn_normal, cbio.df_metadata, 
                                        keep_biotypes=("protein_coding", "lncRNA", "miRNA"),
                                        gene_key="geneid", force=force, verbose=verbose)

#--- reference geneid --> 
gene_map = prism.load_gene_map("geneid")

"""
The reference is the hard half. Peng CRA001160 is symbol-indexed, and it's the pre-2018 symbols we just diagnosed. 
Moving the bulk to ENSG doesn't help unless the reference moves too — otherwise the intersection goes to zero. 
So you still need one symbol→ENSG mapping for the reference, 
and it should use an archived Ensembl release (~92/93, contemporary with Peng), 
not a current service, for the same reason the two-step round-trip was risky.

ENSG IDs need normalising.
"""

In [ ]:
print(dfn_tumor.shape)
dfn_tumor.head(3)

In [ ]:
force=False
verbose=True

fname = "count-matrix.txt"
adata = prism.load_matrix(fname=fname, sep=' ', force=force, verbose=verbose)

fname_ad = fname.replace('.txt', '.h5ad')
filename_ad = prism.root_singc / fname_ad
compression = "gzip"


verbose=True
fname_celltype = "all_celltype.txt"
adata_ct = prism.attach_celltypes(adata=adata, fname_celltype=fname_celltype, verbose=verbose)

ref, s2t = prism.pseudobulk_reference(adata_ct)

print(ref.shape)
ref.head(3)

In [ ]:
bulk = pd.read_csv(prism.root_singc / "bulk_matrix.tsv", sep="\t", index_col=0, usecols=[0])
print("CTGF in bulk:", "CTGF" in bulk.index, "| CCN2 in bulk:", "CCN2" in bulk.index)

In [ ]:
b, r = set(bulk.index), set(ref.columns)
lost = sorted(r - b)
print(f"ref-only: {len(lost)}")

In [ ]:
import mygene

hits = mygene.MyGeneInfo().querymany(lost, scopes="symbol,alias", fields="symbol", species="human", as_dataframe=True)
rec = hits[hits["symbol"].isin(b)]
print(f"recoverable by alias: {len(rec)}")
rec["symbol"].head(20)

In [ ]:
alias_map = dict(zip(rec.index, rec["symbol"]))     # old symbol -> current

ref_e, df_symb_ens = prism.harmonize_reference_to_ensembl(ref, gene_map, alias_map=alias_map)

In [ ]:
df_symb_ens2 = df_symb_ens[~pd.isnull(df_symb_ens.geneid)]
df_symb_ens2

In [ ]:
ref_e.head(2)

In [ ]:
gene_subset = prism.select_genes(ref_e)
len(gene_subset)

In [ ]:
print(type(s2t), len(s2t))
s2t

In [ ]:
verbose=True
force=False

meta_desc = dict(reference="Peng2019_CRA001160",
              cohorts=["TCGA-PAAD", "CPTAC3"],
              strand="unstranded",
              method="InstaPrism")

res = prism.run_bayesprism(df_bulk=df_bulk, ref=ref_e, state_to_type=s2t,
                           gene_subset=gene_subset, meta_desc=meta_desc, force=force)

type(res)

### why Zfull resulted in 16550 genes?

Because full_Z reconstructs the full gene set, not the subset BayesPrism fitted on.

The three numbers you've seen trace it:

- 16550 — genes in full_Z, the whole expression matrix
- 1604 — genes in cell_type_expression, the marker-based fit
- ~10000 — after min_share/min_counts filtering

BayesPrism runs on gene_subset (marker/signature genes) for tractability and identifiability. That gave 1604. 

full_Z then projects the remaining ~15000 genes onto the fitted compartment basis — which is precisely why you built it: to recover the lncRNA/antisense loci (FAM83A-AS1, HOXA10-AS, HOXB-AS3/4, MIR7-3HG) that the marker fit excluded.

Shape is (153 samples, 10 cell types, 16550 genes) — build_ms_from_full_Z resolves that axis order automatically.

The consequence you should hold onto: 
- those ~15000 recovered genes are not Gibbs posterior estimates. 
- they're projections onto a basis fitted from 1604 genes, 
- so their sampling variance is structurally different 
  - no posterior shrinkage in the same sense, 
  - and their between-sample variation partly reflects the projection rather than compartment-specific evidence.

In [ ]:
Zfull, gfull = prism.full_Z(res, df_bulk, ref_e)
Zfull.shape

In [ ]:
Zfull

In [ ]:
dic = {}

for cell_state in res.states:
    Z = prism.state_expression(Zfull, gfull, res, cell_state)
    dic[cell_state] = Z
    print(cell_state, Z.shape)


In [ ]:
i=0
key = list(dic.keys())[i]

print(key)
dic[key]

### Ductal 2 - malignant

In [ ]:
Zmal = prism.state_expression(Zfull, gfull, res, "Ductal cell type 2")
Zmal.shape

In [ ]:
gfull[:3]

In [ ]:
df_symb_ens2.head(3)

In [ ]:
df_bulk.head(3)

In [ ]:
for g in ["FAM83A-AS1", "HOXA10-AS", "HOXB-AS3", "MIR7-3HG"]:

    row = df_symb_ens2[df_symb_ens2['current_symbol'] == g]
    if row.empty:
        print(f"Gene {g} not found in df_symb_ens2")
    else:
        geneid = row.iloc[0].geneid
        print(g, geneid, geneid in df_bulk.index.to_list())

In [ ]:
prog1 = ["FAM83A-AS1", "HOXA10-AS", "HOXB-AS3", "HOXB-AS4", "MIR7-3HG"]
prog2 = ["GATA6", "KRT17", "NEAT1", "H19", "DLEU1", "DLEU2"]

### survived build_bulk_matrix?

> Almost certainly df_bulk is the culprit: build_bulk_matrix defaults to keep_biotypes=("protein_coding",), which removes every lncRNA. Rebuild with them included:

In [ ]:
def find_geneid(symbol:str):
    row = df_symb_ens2[df_symb_ens2['current_symbol'] == symbol]
    if row.empty:
        return None, False
    else:
        geneid = row.iloc[0].geneid
        return row.iloc[0].geneid, geneid in df_bulk.index.to_list()
        

{g: find_geneid(g) for g in prog1}

In [ ]:
ref.head(3)

In [ ]:
print("in bulk gene_map:", (gene_map["symbol"] == "HOXB-AS4").any())
print("in ref (raw):", "HOXB-AS4" in ref.columns)
print("in report:", (df_symb_ens2["ref_symbol"] == "HOXB-AS4").any())

In [ ]:
{g: find_geneid(g) for g in prog2}

### Confirm

NEAT1, H19 and DLEU2 missing is a much stronger signal than HOXB-AS4 was. These aren't obscure — NEAT1 and H19 are among the most abundant lncRNAs in any tissue, H19 is a classic PDAC lncRNA, and NEAT1 is the paraspeckle scaffold. If a reference lacks those, the gap isn't about low abundance.

The likely cause is nuclear retention. NEAT1 and H19 are predominantly nuclear, and Peng used 10x 3' whole-cell scRNA-seq — cytoplasmic-biased, so nuclear-retained transcripts are systematically under-recovered. DLEU2 is the same class. Note DLEU1 survived while DLEU2 didn't, which is consistent with transcript-level capture differences at the same locus.

Confirm it's the reference and not the mapping:

In [ ]:
for g in ["NEAT1","H19","DLEU2","MALAT1","XIST","KCNQ1OT1","MEG3"]:
    print(f"{g:10s} bulk:{(gene_map['symbol']==g).any()!s:5s} ref:{g in ref.columns}")